In [3]:
!pip install openai python-dotenv

In [1]:
import os
import json
import functools

from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIConnectionError

In [3]:
with open(".env", "w") as file:
    file.write("GROQ_API_KEY=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")

print(".env file created successfully!")

.env file created successfully!


In [4]:
import os

print(os.path.exists(".env"))

True


In [5]:
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API Key Loaded:", bool(GROQ_API_KEY))

API Key Loaded: True


In [7]:
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

print("Connected to Groq successfully!")

Connected to Groq successfully!


In [8]:
MODEL = "openai/gpt-oss-20b"

print("Model selected:", MODEL)

Model selected: openai/gpt-oss-20b


In [9]:
BANK = {
    "savings account": "Deposit money and earn interest",
    "current account": "For frequent business transactions",
    "fixed deposit": "Deposit money for a fixed period",
    "upi": "Instant money transfer",
    "loans": "Borrow money from the bank",
    "debit card": "For payments and cash withdrawals"
}

print("Banking knowledge base created successfully!")

Banking knowledge base created successfully!


In [10]:
SYSTEM = f"""
ROLE: You are a friendly assistant for ABC Bank.

FACTS:
Banking services: {json.dumps(BANK)}

RULES:
1. Never ask for PIN, CVV, OTP or passwords.
2. Never invent account details or interest rates.
3. If information is unknown, say "Please contact bank support."
4. Remember the conversation.
5. Answer in English or Telugu matching the customer.

FORMAT:
Keep responses under 50 words.
Be polite and helpful.
"""

print("System prompt created successfully!")

System prompt created successfully!


In [11]:
def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        result = func(*args, **kwargs)

        print(f"[log] {func.__name__}{args} -> {result!r}")

        return result

    return wrapper

In [12]:
@log_call
def get_info(item: str) -> str:

    return BANK.get(
        item.lower().strip(),
        "Information not available."
    )

In [13]:
print("UPI:",get_info("upi"))
print("Loans:", get_info("loans"))
print("Savings Account:", get_info("savings account"))

[log] get_info('upi',) -> 'Instant money transfer'
UPI: Instant money transfer
[log] get_info('loans',) -> 'Borrow money from the bank'
Loans: Borrow money from the bank
[log] get_info('savings account',) -> 'Deposit money and earn interest'
Savings Account: Deposit money and earn interest


In [14]:
history = [
    {
        "role": "system",
        "content": SYSTEM
    }
]

MAX_TURNS = 8

print("Conversation history initialized!")

Conversation history initialized!


In [15]:
def chat(question: str) -> str:

    history.append({
        "role": "user",
        "content": question
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:

        reply = "Please try again in a minute."

    except APIConnectionError:

        reply = "Server connection failed. Try again."

    except Exception as e:

        reply = "Something went wrong. Please try again."
        print("Error:", e)

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [ ]:
print("Welcome to ABC Bank Customer Support!")
print("Type 'bye' to exit.\n")

while True:

    user_input = input("You: ")

    if user_input.lower().strip() in ("bye", "exit", "quit"):

        print("Bot: Thank you for contacting ABC Bank!")
        break

    print("Bot:", chat(user_input))

Welcome to ABC Bank Customer Support!
Type 'bye' to exit.



You:  What is your main purpose


Error: Decompressor.decompress() got an unexpected keyword argument 'output_buffer_limit'
Bot: Something went wrong. Please try again.
